# Chapitre 4 - Donnees & Feature Engineering
### Memoire M2 ISF - Modelisation du Spread de Taux

Ce notebook couvre :
1. Telechargement des donnees BCE
2. Calcul du spread (variable cible)
3. Feature engineering
4. Statistiques descriptives
5. Visualisations

---

## 0. Imports

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'requests', 'pandas', 'numpy',
                'matplotlib', 'seaborn', 'scipy', '-q'])

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from io import StringIO
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('figures', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('OK imports')

## 1. Telechargement des donnees BCE

| Serie | Identifiant BCE | Role |
|-------|----------------|------|
| Euribor 3M | FM/M.U2.EUR.RT0.MM.EURIBOR3MD_.HSTA | Taux court terme |
| Euribor 6M | FM/M.U2.EUR.RT0.MM.EURIBOR6MD_.HSTA | Taux court terme alt |
| OIS 1M | FM/M.U2.EUR.RT0.MM.EURIBOR1MD_.HSTA | Proxy sans risque |
| Swap 1Y | YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y | Courbe court |
| Swap 2Y | YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y | Courbe moyen |
| Swap 5Y | YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y | Numerateur spread |
| Swap 10Y | YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y | Courbe long |

In [ ]:
def fetch_ecb(series_key, name, start='2000-01-01'):
    """
    Telecharge une serie depuis l'API BCE.
    Retourne une pd.Series avec index datetime mensuel.
    """
    flow, key = series_key.split('/', 1)
    url = (
        f'https://data-api.ecb.europa.eu/service/data/{flow}/{key}'
        f'?startPeriod={start}&format=csvdata'
    )
    print(f'  Telechargement : {name} ...')
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text))
        df['date'] = pd.to_datetime(df['TIME_PERIOD'])
        df = df.set_index('date').sort_index()
        series = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
        series.name = name
        print(f'  OK {name} : {len(series)} obs ({series.index[0].year}-{series.index[-1].year})')
        return series
    except Exception as e:
        print(f'  ERREUR {name} : {e}')
        return None

print('Fonction fetch_ecb definie')

In [ ]:
euribor_3m = fetch_ecb('FM/M.U2.EUR.RT0.MM.EURIBOR3MD_.HSTA', 'Euribor_3M')
euribor_6m = fetch_ecb('FM/M.U2.EUR.RT0.MM.EURIBOR6MD_.HSTA', 'Euribor_6M')
ois        = fetch_ecb('FM/M.U2.EUR.RT0.MM.EURIBOR1MD_.HSTA',  'OIS_1M')
swap_1y    = fetch_ecb('YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y',  'Swap_1Y')
swap_2y    = fetch_ecb('YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y',  'Swap_2Y')
swap_5y    = fetch_ecb('YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y',  'Swap_5Y')
swap_10y   = fetch_ecb('YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y', 'Swap_10Y')
print('Telechargement termine !')

In [ ]:
series_list = [s for s in [euribor_3m, euribor_6m, ois,
                            swap_1y, swap_2y, swap_5y, swap_10y]
               if s is not None]

df_raw = pd.concat(series_list, axis=1)
df_raw = df_raw.resample('ME').last()
df_raw = df_raw.dropna(thresh=4)
df_raw.to_csv('data/raw/taux_bruts.csv')

print(f'Dimensions : {df_raw.shape}')
print(f'Periode    : {df_raw.index[0].strftime("%Y-%m")} -> {df_raw.index[-1].strftime("%Y-%m")}')
df_raw.tail()

## 2. Feature Engineering

**Variable cible :** Spread = Swap5Y - Euribor3M

**Features explicatives :**

| Feature | Formule | Interpretation |
|---------|---------|----------------|
| Level | Euribor 3M | Niveau des taux (politique BCE) |
| Slope | Swap10Y - Euribor3M | Pente de la courbe (beta2 DNS) |
| Curvature | 2*Swap5Y - Swap10Y - Euribor3M | Courbure (beta3 DNS) |
| Volatility | std(Euribor3M, 3 mois) | Incertitude / stress recent |
| Delta_Rate | Euribor3M(t) - Euribor3M(t-1) | Acceleration politique monetaire |
| Spread_lag1/3/6 | Spread decale | Autoregressivite |
| OIS_Spread | Swap2Y - OIS1M | Prime de liquidite |

In [ ]:
feat = pd.DataFrame(index=df_raw.index)

# Variable cible
feat['Spread']      = df_raw['Swap_5Y'] - df_raw['Euribor_3M']

# Features
feat['Level']       = df_raw['Euribor_3M']
feat['Slope']       = df_raw['Swap_10Y'] - df_raw['Euribor_3M']
feat['Curvature']   = 2*df_raw['Swap_5Y'] - df_raw['Swap_10Y'] - df_raw['Euribor_3M']
feat['Volatility']  = df_raw['Euribor_3M'].rolling(window=3).std()
feat['Delta_Rate']  = df_raw['Euribor_3M'].diff(1)
feat['Spread_lag1'] = feat['Spread'].shift(1)
feat['Spread_lag3'] = feat['Spread'].shift(3)
feat['Spread_lag6'] = feat['Spread'].shift(6)

if 'OIS_1M' in df_raw.columns and 'Swap_2Y' in df_raw.columns:
    feat['OIS_Spread'] = df_raw['Swap_2Y'] - df_raw['OIS_1M']

feat = feat.dropna()
feat.to_csv('data/processed/dataset_final.csv')

print(f'Dataset final : {feat.shape[0]} lignes x {feat.shape[1]} colonnes')
feat.head()

## 3. Statistiques Descriptives

In [ ]:
stats = feat.describe().T[['mean','std','min','25%','50%','75%','max']]
stats.columns = ['Moyenne','Std','Min','Q25','Mediane','Q75','Max']
stats.to_csv('data/processed/statistiques.csv')
print(stats.round(4).to_string())

print(f'Spread moyen : {feat["Spread"].mean():.3f}%')
print(f'Spread min   : {feat["Spread"].min():.3f}% ({feat["Spread"].idxmin().strftime("%Y-%m")})')
print(f'Spread max   : {feat["Spread"].max():.3f}% ({feat["Spread"].idxmax().strftime("%Y-%m")})')
print(f'% positif    : {(feat["Spread"] > 0).mean()*100:.1f}%')

## 4. Visualisations

### Figure 1 : Evolution historique du spread

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(feat.index, feat['Spread'], color='steelblue', linewidth=1.5,
        label='Spread Swap 5Y - Euribor 3M')
ax.fill_between(feat.index, feat['Spread'], 0,
                where=(feat['Spread'] > 0), alpha=0.15, color='steelblue')
ax.fill_between(feat.index, feat['Spread'], 0,
                where=(feat['Spread'] < 0), alpha=0.30, color='red')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

episodes = [
    ('2008-09-01', 'Faillite Lehman', 'red'),
    ('2012-06-01', 'Crise Souveraine', 'darkorange'),
    ('2020-03-01', 'COVID-19', 'purple'),
    ('2022-07-01', 'Hausse BCE', 'red'),
]
y_max = feat['Spread'].max()
for date_str, label, color in episodes:
    ts = pd.Timestamp(date_str)
    if feat.index[0] <= ts <= feat.index[-1]:
        ax.axvline(ts, color=color, linewidth=1.5, linestyle=':', alpha=0.8)
        ax.annotate(label, xy=(ts, y_max*0.75), fontsize=8.5, color=color,
                    ha='center', bbox=dict(boxstyle='round,pad=0.2',
                    facecolor='white', alpha=0.7))

ax.set_title('Spread de Taux Swap EUR (Swap 5Y - Euribor 3M)')
ax.set_ylabel('Spread (%)')
ax.set_xlabel('Date')
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('figures/fig1_spread.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK fig1')

### Figure 2 : Facteurs Nelson-Siegel Dynamique

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)
factors = [
    ('Level',     'Beta1 - Niveau (Euribor 3M)',           'steelblue'),
    ('Slope',     'Beta2 - Pente (Swap10Y - Euribor3M)',   'darkgreen'),
    ('Curvature', 'Beta3 - Courbure (2xSwap5Y-10Y-3M)',    'darkorange'),
]
for ax, (col, label, color) in zip(axes, factors):
    if col in feat.columns:
        ax.plot(feat.index, feat[col], color=color, linewidth=1.3)
        ax.fill_between(feat.index, feat[col], 0, alpha=0.12, color=color)
        ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
        ax.set_ylabel(label, fontsize=10)
axes[0].set_title('Facteurs de Nelson-Siegel Dynamique (Diebold & Li, 2006)')
axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('figures/fig2_nelson_siegel.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK fig2')

### Figure 3 : Distribution + scatter + volatilite

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
feat['Spread'].hist(bins=50, ax=ax, color='steelblue',
                    edgecolor='white', density=True, alpha=0.8)
ax.axvline(feat['Spread'].mean(), color='black', linewidth=2,
           label=f'Moy={feat["Spread"].mean():.2f}%')
ax.axvline(feat['Spread'].quantile(0.05), color='red', linewidth=2,
           linestyle='--', label=f'5e pct={feat["Spread"].quantile(0.05):.2f}%')
ax.axvline(feat['Spread'].quantile(0.95), color='green', linewidth=2,
           linestyle='--', label=f'95e pct={feat["Spread"].quantile(0.95):.2f}%')
ax.set_title('Distribution du Spread')
ax.set_xlabel('Spread (%)')
ax.legend(fontsize=8)

ax = axes[1]
sc = ax.scatter(feat['Slope'], feat['Spread'],
                c=feat.index.year, cmap='RdYlBu_r', alpha=0.6, s=20)
plt.colorbar(sc, ax=ax, label='Annee')
ax.set_xlabel('Pente (%)')
ax.set_ylabel('Spread (%)')
ax.set_title('Spread vs Pente')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')

ax = axes[2]
rolling_vol = feat['Spread'].rolling(12).std()
ax.plot(feat.index, rolling_vol, color='firebrick', linewidth=1.3)
ax.fill_between(feat.index, rolling_vol, 0, alpha=0.15, color='firebrick')
ax.set_title('Volatilite rolling Spread (12 mois)')
ax.set_xlabel('Date')
ax.set_ylabel('Volatilite (%)')
ax.xaxis.set_major_locator(mdates.YearLocator(4))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('figures/fig3_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK fig3')

### Figure 4 : Matrice de correlation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = feat.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Matrice de Correlation des Features')
plt.tight_layout()
plt.savefig('figures/fig4_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK fig4')

### Figure 5 : Regimes de marche

In [ ]:
def classify_regime(row):
    vol   = row.get('Volatility', 0)
    level = row.get('Level', 0)
    if vol > feat['Volatility'].quantile(0.75):
        return 'Stress'
    elif level < 0.5:
        return 'Taux bas'
    elif level > 2.0:
        return 'Taux eleves'
    else:
        return 'Transition'

feat['Regime'] = feat.apply(classify_regime, axis=1)
print(feat['Regime'].value_counts())
print(feat.groupby('Regime')['Spread'].agg(['mean','std']).round(3))

In [ ]:
regime_colors = {
    'Taux eleves': 'steelblue',
    'Taux bas':    'darkgreen',
    'Stress':      'firebrick',
    'Transition':  'darkorange'
}
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

ax = axes[0]
ax.plot(feat.index, feat['Spread'], color='gray', linewidth=0.8, alpha=0.5)
for regime, color in regime_colors.items():
    mask = feat['Regime'] == regime
    if mask.any():
        ax.scatter(feat.index[mask], feat['Spread'][mask],
                   color=color, s=15, alpha=0.8, label=regime)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Spread par Regime de Marche')
ax.set_ylabel('Spread (%)')
ax.legend(fontsize=9)

ax = axes[1]
for regime, color in regime_colors.items():
    mask = feat['Regime'] == regime
    if mask.sum() > 5:
        feat['Spread'][mask].plot.kde(ax=ax, color=color, linewidth=2, label=regime)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Distribution du Spread par Regime')
ax.set_xlabel('Spread (%)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig5_regimes.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK fig5')

## 5. Resume

In [ ]:
print('CHAPITRE 4 TERMINE')
print(f'Dataset : {feat.shape[0]} observations, {feat.shape[1]} colonnes')
print(f'Variable cible - Spread moyen : {feat["Spread"].mean():.3f}%')
print('Fichiers generes :')
print('  data/raw/taux_bruts.csv')
print('  data/processed/dataset_final.csv')
print('  figures/fig1 a fig5')
print('Prochaine etape : Chapitre 5 - XGBoost + validation temporelle')